In [20]:
# ============================================
# BLOCK 1: IMPORTS AND SETUP
# ============================================
# WHAT THIS DOES:
# - Imports all necessary libraries
# - Sets up visualization style
# - Checks if SMOTE is available
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Check if SMOTE is available
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
    print("✅ SMOTE is available!")
except ImportError:
    SMOTE_AVAILABLE = False
    print("⚠️ SMOTE not available. Will use class weights instead.")

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("\n" + "="*60)
print("FEATURE ENGINEERING NOTEBOOK")
print("="*60)
print(f"SMOTE Available: {SMOTE_AVAILABLE}")

✅ SMOTE is available!

FEATURE ENGINEERING NOTEBOOK
SMOTE Available: True


In [21]:
# ============================================
# BLOCK 2: LOAD THE DATASET
# ============================================
# WHAT THIS DOES:
# - Loads the dataset from the data folder
# - Creates a copy to preserve original data
# - Shows initial shape and columns
# ============================================

# Try multiple possible file paths
possible_paths = [
    '../data/CVD Dataset.csv',
    '../data/CVD_Dataset.csv',
    'data/CVD Dataset.csv',
    'data/CVD_Dataset.csv',
]

file_loaded = False
for path in possible_paths:
    if os.path.exists(path):
        print(f"✅ Found file at: {path}")
        df = pd.read_csv(path)
        file_loaded = True
        break

if not file_loaded:
    raise FileNotFoundError("Could not find CVD Dataset.csv in any expected location!")

# Create a copy to work with
df_original = df.copy()
df = df.copy()

print("\n" + "="*60)
print("DATASET LOADED SUCCESSFULLY!")
print("="*60)
print(f"📊 Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"📋 Columns: {df.columns.tolist()}")

✅ Found file at: ../data/CVD Dataset.csv

DATASET LOADED SUCCESSFULLY!
📊 Shape: 1529 rows, 22 columns
📋 Columns: ['Sex', 'Age', 'Weight (kg)', 'Height (m)', 'BMI', 'Abdominal Circumference (cm)', 'Blood Pressure (mmHg)', 'Total Cholesterol (mg/dL)', 'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 'Smoking Status', 'Diabetes Status', 'Physical Activity Level', 'Family History of CVD', 'Height (cm)', 'Waist-to-Height Ratio', 'Systolic BP', 'Diastolic BP', 'Blood Pressure Category', 'Estimated LDL (mg/dL)', 'CVD Risk Score', 'CVD Risk Level']


In [22]:
# ============================================
# BLOCK 3: HANDLE MISSING VALUES
# ============================================
# WHAT THIS DOES:
# - Identifies all columns with missing values
# - Fills numerical columns with MEDIAN
# - Fills categorical columns with MODE
# - Drops rows where target is missing
# - Shows before/after comparison
# ============================================

print("\n" + "="*60)
print("HANDLING MISSING VALUES")
print("="*60)

# Check missing values before
missing_before = df.isnull().sum().sum()
print(f"\n❌ Missing values BEFORE: {missing_before}")

# Identify numerical and categorical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Remove target from categorical if present
target_col = 'CVD Risk Level'
if target_col in categorical_cols:
    categorical_cols.remove(target_col)

# Remove columns that are already encoded or derived
exclude_cols = ['Blood Pressure (mmHg)', 'Blood Pressure Category']
categorical_cols = [col for col in categorical_cols if col not in exclude_cols]

print(f"\n📊 Numerical columns: {len(numerical_cols)}")
print(f"📊 Categorical columns: {len(categorical_cols)}")

# Fill numerical columns with median
print("\n🔧 Filling numerical columns with MEDIAN...")
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"   ✅ {col}: filled with median {median_val:.2f}")

# Fill categorical columns with mode
print("\n🔧 Filling categorical columns with MODE...")
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"   ✅ {col}: filled with mode '{mode_val}'")

# Drop rows where target is missing (if any)
if target_col in df.columns and df[target_col].isnull().sum() > 0:
    print(f"\n⚠️ Dropping {df[target_col].isnull().sum()} rows with missing target...")
    df = df.dropna(subset=[target_col])

# Check missing values after
missing_after = df.isnull().sum().sum()
print(f"\n✅ Missing values AFTER: {missing_after}")
print(f"   ✅ All missing values handled!")


HANDLING MISSING VALUES

❌ Missing values BEFORE: 1022

📊 Numerical columns: 14
📊 Categorical columns: 5

🔧 Filling numerical columns with MEDIAN...
   ✅ Age: filled with median 46.00
   ✅ Weight (kg): filled with median 86.61
   ✅ Height (m): filled with median 1.75
   ✅ BMI: filled with median 28.16
   ✅ Abdominal Circumference (cm): filled with median 91.60
   ✅ Total Cholesterol (mg/dL): filled with median 197.00
   ✅ HDL (mg/dL): filled with median 56.00
   ✅ Fasting Blood Sugar (mg/dL): filled with median 115.00
   ✅ Height (cm): filled with median 175.69
   ✅ Waist-to-Height Ratio: filled with median 0.52
   ✅ Systolic BP: filled with median 125.00
   ✅ Diastolic BP: filled with median 82.00
   ✅ Estimated LDL (mg/dL): filled with median 109.00
   ✅ CVD Risk Score: filled with median 16.88

🔧 Filling categorical columns with MODE...

✅ Missing values AFTER: 0
   ✅ All missing values handled!


In [23]:
# ============================================
# BLOCK 4: REMOVE REDUNDANT COLUMNS
# ============================================
# WHAT THIS DOES:
# - Removes Height (cm) because we have Height (m)
# - Removes Estimated LDL because it's derived from Total Cholesterol
# - KEEPS Waist-to-Height Ratio (clinically important!)
# ============================================

print("\n" + "="*60)
print("REMOVING REDUNDANT COLUMNS")
print("="*60)

print("\n❌ Columns to REMOVE:")
print("   1. Height (cm) - same as Height (m)")
print("   2. Estimated LDL (mg/dL) - derived from Total Cholesterol")

# Columns to drop
columns_to_drop = ['Height (cm)', 'Estimated LDL (mg/dL)']

# Check if columns exist before dropping
for col in columns_to_drop:
    if col in df.columns:
        print(f"   ✅ Removing: {col}")
        df = df.drop(columns=[col])
    else:
        print(f"   ⚠️ Column not found: {col}")

print(f"\n✅ Remaining columns: {len(df.columns)}")


REMOVING REDUNDANT COLUMNS

❌ Columns to REMOVE:
   1. Height (cm) - same as Height (m)
   2. Estimated LDL (mg/dL) - derived from Total Cholesterol
   ✅ Removing: Height (cm)
   ✅ Removing: Estimated LDL (mg/dL)

✅ Remaining columns: 20


In [24]:
# ============================================
# BLOCK 5: CREATE NEW CLINICAL FEATURES
# ============================================
# WHAT THIS DOES:
# - Creates PULSE PRESSURE (Systolic - Diastolic)
# - Creates CHOLESTEROL RATIO (Total Cholesterol / HDL)
# - Creates BMI CATEGORY (Underweight, Normal, Overweight, Obese)
# - Creates AGE GROUP (Young, Adult, Middle, Senior, Elderly)
# - Creates OBESITY RISK (1 if Obese, else 0)
# - Creates HYPERTENSION RISK (1 if High BP, else 0)
# - Creates METABOLIC SCORE (sum of risk factors)
# - Creates PHYSICAL ACTIVITY SCORE (numeric conversion)
# ============================================

print("\n" + "="*60)
print("CREATING NEW FEATURES")
print("="*60)

# 1. Pulse Pressure
print("\n🔧 Creating: Pulse Pressure")
df['Pulse Pressure'] = df['Systolic BP'] - df['Diastolic BP']
print(f"   ✅ Added: Pulse Pressure (mean: {df['Pulse Pressure'].mean():.1f})")

# 2. Cholesterol Ratio
print("\n🔧 Creating: Cholesterol Ratio")
df['Cholesterol_HDL_Ratio'] = df['Total Cholesterol (mg/dL)'] / df['HDL (mg/dL)']
print(f"   ✅ Added: Cholesterol_HDL_Ratio (mean: {df['Cholesterol_HDL_Ratio'].mean():.1f})")

# 3. BMI Category
print("\n🔧 Creating: BMI Category")
def get_bmi_category(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif bmi < 25:
        return 'Normal'
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

df['BMI_Category'] = df['BMI'].apply(get_bmi_category)
print(f"   ✅ Added: BMI_Category")
print(f"      Distribution:\n{df['BMI_Category'].value_counts()}")

# 4. Age Group
print("\n🔧 Creating: Age Group")
def get_age_group(age):
    if age < 35:
        return 'Young'
    elif age < 50:
        return 'Adult'
    elif age < 60:
        return 'Middle'
    elif age < 70:
        return 'Senior'
    else:
        return 'Elderly'

df['Age_Group'] = df['Age'].apply(get_age_group)
print(f"   ✅ Added: Age_Group")
print(f"      Distribution:\n{df['Age_Group'].value_counts()}")

# 5. Obesity Risk
print("\n🔧 Creating: Obesity Risk")
df['Obesity_Risk'] = (df['BMI'] >= 30).astype(int)
print(f"   ✅ Added: Obesity_Risk (Obese: {df['Obesity_Risk'].sum()} patients)")

# 6. Hypertension Risk
print("\n🔧 Creating: Hypertension Risk")
df['Hypertension_Risk'] = ((df['Systolic BP'] >= 140) | (df['Diastolic BP'] >= 90)).astype(int)
print(f"   ✅ Added: Hypertension_Risk (Hypertensive: {df['Hypertension_Risk'].sum()} patients)")

# 7. Metabolic Score
print("\n🔧 Creating: Metabolic Score")
df['Metabolic_Score'] = (
    df['Obesity_Risk'] + 
    df['Hypertension_Risk'] + 
    (df['Total Cholesterol (mg/dL)'] >= 240).astype(int) +
    (df['Fasting Blood Sugar (mg/dL)'] >= 126).astype(int) +
    (df['Smoking Status'] == 'Y').astype(int)
)
print(f"   ✅ Added: Metabolic_Score (mean: {df['Metabolic_Score'].mean():.1f})")

# 8. Physical Activity Score
print("\n🔧 Creating: Physical Activity Score")
activity_map = {'Low': 0, 'Moderate': 1, 'High': 2}
df['Physical_Activity_Score'] = df['Physical Activity Level'].map(activity_map)
print(f"   ✅ Added: Physical_Activity_Score")

# 9. Blood Pressure Category (derive if missing)
print("\n🔧 Creating/Verifying: Blood Pressure Category")
def get_bp_category(systolic, diastolic):
    if systolic < 120 and diastolic < 80:
        return 'Normal'
    elif systolic < 130 and diastolic < 80:
        return 'Elevated'
    elif systolic < 140 or diastolic < 90:
        return 'Hypertension Stage 1'
    else:
        return 'Hypertension Stage 2'

if 'Blood Pressure Category' not in df.columns or df['Blood Pressure Category'].isnull().sum() > 0:
    df['Blood Pressure Category'] = df.apply(
        lambda row: get_bp_category(row['Systolic BP'], row['Diastolic BP']), axis=1
    )
    print(f"   ✅ Added/Updated: Blood Pressure Category")
else:
    print(f"   ✅ Already exists: Blood Pressure Category")

print(f"\n📊 Blood Pressure Category Distribution:")
print(df['Blood Pressure Category'].value_counts())

print(f"\n✅ Total new features added: 9")


CREATING NEW FEATURES

🔧 Creating: Pulse Pressure
   ✅ Added: Pulse Pressure (mean: 42.7)

🔧 Creating: Cholesterol Ratio
   ✅ Added: Cholesterol_HDL_Ratio (mean: 3.8)

🔧 Creating: BMI Category
   ✅ Added: BMI_Category
      Distribution:
BMI_Category
Obese          635
Normal         421
Overweight     379
Underweight     94
Name: count, dtype: int64

🔧 Creating: Age Group
   ✅ Added: Age_Group
      Distribution:
Age_Group
Adult      676
Middle     369
Young      268
Senior     126
Elderly     90
Name: count, dtype: int64

🔧 Creating: Obesity Risk
   ✅ Added: Obesity_Risk (Obese: 635 patients)

🔧 Creating: Hypertension Risk
   ✅ Added: Hypertension_Risk (Hypertensive: 748 patients)

🔧 Creating: Metabolic Score
   ✅ Added: Metabolic_Score (mean: 2.1)

🔧 Creating: Physical Activity Score
   ✅ Added: Physical_Activity_Score

🔧 Creating/Verifying: Blood Pressure Category
   ✅ Already exists: Blood Pressure Category

📊 Blood Pressure Category Distribution:
Blood Pressure Category
Hyperten

In [25]:
# ============================================
# BLOCK 6: ENCODE CATEGORICAL VARIABLES
# ============================================
# WHAT THIS DOES:
# - Converts text categories to numbers
# - Binary encoding: Sex, Smoking, Diabetes, Family History
# - Ordinal encoding: Physical Activity, Blood Pressure
# - Label Encoding: Target (CVD Risk Level)
# - One-Hot Encoding: BMI Category, Age Group
# ============================================

print("\n" + "="*60)
print("ENCODING CATEGORICAL VARIABLES")
print("="*60)

# Initialize label encoder
le = LabelEncoder()

# 1. Binary Encoding (Y/N, M/F)
print("\n🔧 Binary Encoding (Y/N, M/F):")
binary_cols = ['Sex', 'Smoking Status', 'Diabetes Status', 'Family History of CVD']
for col in binary_cols:
    if col in df.columns:
        df[f'{col}_Encoded'] = le.fit_transform(df[col].astype(str))
        print(f"   ✅ {col}: {df[col].unique()} → {df[f'{col}_Encoded'].unique()}")

# 2. Ordinal Encoding (Physical Activity Level)
print("\n🔧 Ordinal Encoding (Physical Activity):")
if 'Physical Activity Level' in df.columns:
    activity_map = {'Low': 0, 'Moderate': 1, 'High': 2}
    df['Physical_Activity_Encoded'] = df['Physical Activity Level'].map(activity_map)
    print(f"   ✅ Physical Activity Level: Low=0, Moderate=1, High=2")

# 3. Ordinal Encoding (Blood Pressure Category)
print("\n🔧 Ordinal Encoding (Blood Pressure):")
if 'Blood Pressure Category' in df.columns:
    bp_map = {
        'Normal': 0, 
        'Elevated': 1, 
        'Hypertension Stage 1': 2, 
        'Hypertension Stage 2': 3
    }
    df['Blood_Pressure_Encoded'] = df['Blood Pressure Category'].map(bp_map)
    print(f"   ✅ Blood Pressure Category: Normal=0, Elevated=1, Stage1=2, Stage2=3")

# 4. Target Encoding (CVD Risk Level)
print("\n🔧 Target Encoding (CVD Risk Level):")
if 'CVD Risk Level' in df.columns:
    risk_map = {'Low': 0, 'INTERMEDIARY': 1, 'HIGH': 2}
    df['CVD_Risk_Encoded'] = df['CVD Risk Level'].map(risk_map)
    print(f"   ✅ CVD Risk Level: Low=0, Intermediate=1, High=2")
    print(f"      Distribution:\n{df['CVD_Risk_Encoded'].value_counts()}")

# 5. One-Hot Encoding (BMI Category, Age Group)
print("\n🔧 One-Hot Encoding (BMI Category, Age Group):")
columns_to_encode = []
if 'BMI_Category' in df.columns:
    columns_to_encode.append('BMI_Category')
if 'Age_Group' in df.columns:
    columns_to_encode.append('Age_Group')

if columns_to_encode:
    df = pd.get_dummies(df, columns=columns_to_encode, prefix=['BMI', 'Age'])
    print(f"   ✅ One-Hot encoded: {columns_to_encode}")
else:
    print("   ⚠️ No columns to one-hot encode")

print(f"\n✅ All categorical variables encoded!")
print(f"📋 Current columns count: {len(df.columns)}")


ENCODING CATEGORICAL VARIABLES

🔧 Binary Encoding (Y/N, M/F):
   ✅ Sex: ['F' 'M'] → [0 1]
   ✅ Smoking Status: ['N' 'Y'] → [0 1]
   ✅ Diabetes Status: ['Y' 'N'] → [1 0]
   ✅ Family History of CVD: ['N' 'Y'] → [0 1]

🔧 Ordinal Encoding (Physical Activity):
   ✅ Physical Activity Level: Low=0, Moderate=1, High=2

🔧 Ordinal Encoding (Blood Pressure):
   ✅ Blood Pressure Category: Normal=0, Elevated=1, Stage1=2, Stage2=3

🔧 Target Encoding (CVD Risk Level):
   ✅ CVD Risk Level: Low=0, Intermediate=1, High=2
      Distribution:
CVD_Risk_Encoded
2.0    728
1.0    581
Name: count, dtype: int64

🔧 One-Hot Encoding (BMI Category, Age Group):
   ✅ One-Hot encoded: ['BMI_Category', 'Age_Group']

✅ All categorical variables encoded!
📋 Current columns count: 42


In [26]:
# ============================================
# BLOCK 7: SCALE NUMERICAL FEATURES
# ============================================
# WHAT THIS DOES:
# - Normalizes numerical features to same scale (mean=0, std=1)
# - Uses StandardScaler: (x - mean) / standard deviation
# ============================================

print("\n" + "="*60)
print("SCALING NUMERICAL FEATURES")
print("="*60)

# Identify numerical columns to scale
scale_cols = [
    'Age', 'Weight (kg)', 'Height (m)', 'BMI', 
    'Abdominal Circumference (cm)', 'Total Cholesterol (mg/dL)', 
    'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 
    'Systolic BP', 'Diastolic BP', 'CVD Risk Score',
    'Pulse Pressure', 'Cholesterol_HDL_Ratio', 'Metabolic_Score',
    'Physical_Activity_Score'
]

# Filter columns that exist
scale_cols = [col for col in scale_cols if col in df.columns]

print(f"\n📊 Scaling {len(scale_cols)} numerical features...")

# Initialize scaler
scaler = StandardScaler()

# Scale the features
df_scaled = df.copy()
df_scaled[scale_cols] = scaler.fit_transform(df[scale_cols])

print("\n📊 Before Scaling (Age example):")
print(f"   Mean: {df['Age'].mean():.2f}, Std: {df['Age'].std():.2f}")
print("\n📊 After Scaling (Age example):")
print(f"   Mean: {df_scaled['Age'].mean():.2f}, Std: {df_scaled['Age'].std():.2f}")

# Use scaled version
df = df_scaled
print("\n✅ Features scaled successfully!")


SCALING NUMERICAL FEATURES

📊 Scaling 15 numerical features...

📊 Before Scaling (Age example):
   Mean: 46.97, Std: 12.10

📊 After Scaling (Age example):
   Mean: 0.00, Std: 1.00

✅ Features scaled successfully!


In [31]:
# ============================================
# BLOCK 8: HANDLE CLASS IMBALANCE (COMPLETE FIX)
# ============================================

print("\n" + "="*60)
print("HANDLING CLASS IMBALANCE")
print("="*60)

# Step 1: Check for string columns in X
print("\n🔍 Checking for string columns in features...")

string_cols = []
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['CVD Risk Level', 'CVD_Risk_Encoded']:
        string_cols.append(col)

if string_cols:
    print(f"⚠️ Found string columns: {string_cols}")
    print("   These need to be encoded before SMOTE!")
    
    # Encode them using LabelEncoder
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    
    for col in string_cols:
        df[col] = le.fit_transform(df[col].astype(str))
        print(f"   ✅ Encoded: {col}")
else:
    print("   ✅ No string columns found")

# Step 2: Recreate CVD_Risk_Encoded from original column
print("\n🔧 Recreating target encoding...")

if 'CVD Risk Level' in df.columns:
    risk_map = {'Low': 0, 'INTERMEDIARY': 1, 'HIGH': 2}
    df['CVD_Risk_Encoded'] = df['CVD Risk Level'].map(risk_map)
    
    # Check for unmapped values
    if df['CVD_Risk_Encoded'].isnull().sum() > 0:
        print(f"⚠️ Found {df['CVD_Risk_Encoded'].isnull().sum()} unmapped values. Dropping...")
        df = df.dropna(subset=['CVD_Risk_Encoded'])
        print(f"   New shape: {df.shape}")

# Step 3: Remove duplicate columns
print("\n🔧 Checking for duplicate columns...")
if df.columns.duplicated().any():
    print(f"   ⚠️ Found duplicate columns. Removing...")
    df = df.loc[:, ~df.columns.duplicated()]
    print(f"   ✅ Removed duplicates. Now {len(df.columns)} columns")
else:
    print("   ✅ No duplicate columns found")

# Step 4: Prepare features and target (ALL NUMERIC NOW)
print("\n🔧 Preparing features and target...")
feature_cols = [col for col in df.columns if col not in ['CVD Risk Level', 'CVD_Risk_Encoded']]
X = df[feature_cols]
y = df['CVD_Risk_Encoded']

# Verify all columns are numeric
print(f"\n📊 Features shape: {X.shape}")
print(f"📊 Target shape: {y.shape}")

# Check if any columns are still strings
still_string = []
for col in X.columns:
    if X[col].dtype == 'object':
        still_string.append(col)

if still_string:
    print(f"⚠️ Still have string columns: {still_string}")
    for col in still_string:
        X[col] = pd.to_numeric(X[col], errors='coerce')
        if X[col].isnull().sum() > 0:
            X[col].fillna(X[col].median(), inplace=True)
    print("   ✅ All columns now numeric")

print(f"\n📊 Target distribution:\n{y.value_counts().sort_index()}")

# Step 5: Check for NaN in y
if y.isnull().sum() > 0:
    print(f"⚠️ Found {y.isnull().sum()} NaN values in target. Dropping...")
    valid_idx = y.notna()
    X = X[valid_idx]
    y = y[valid_idx]
    print(f"   New shape: {X.shape}")

# Step 6: Check if we have at least 2 classes for SMOTE
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
    print("\n✅ SMOTE is available")
except ImportError:
    SMOTE_AVAILABLE = False
    print("\n⚠️ SMOTE not available")

if len(y.unique()) >= 2:
    min_class_count = y.value_counts().min()
    print(f"\n📊 Minimum class count: {min_class_count}")
    
    if SMOTE_AVAILABLE and min_class_count >= 2:
        print("\n🔧 Applying SMOTE...")
        smote = SMOTE(random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X, y)
        
        print(f"\n✅ After SMOTE:")
        print(f"   Shape: {X_resampled.shape}")
        print(f"   Target distribution:\n{pd.Series(y_resampled).value_counts().sort_index()}")
        
        X_final = X_resampled
        y_final = y_resampled
        class_weight_dict = None
    else:
        print("\n🔧 Using class weights (SMOTE not applicable)...")
        from sklearn.utils.class_weight import compute_class_weight
        
        classes = np.unique(y)
        class_weights = compute_class_weight('balanced', classes=classes, y=y)
        class_weight_dict = dict(zip(classes, class_weights))
        
        print(f"\n📊 Class weights:")
        for cls, weight in class_weight_dict.items():
            risk_name = {0: 'LOW', 1: 'INTERMEDIARY', 2: 'HIGH'}.get(cls, f'Class {cls}')
            print(f"   {risk_name}: {weight:.2f}")
        
        X_final = X
        y_final = y
else:
    print(f"\n⚠️ Only {len(y.unique())} class found. Using class weights...")
    from sklearn.utils.class_weight import compute_class_weight
    
    classes = np.unique(y)
    class_weights = compute_class_weight('balanced', classes=classes, y=y)
    class_weight_dict = dict(zip(classes, class_weights))
    
    print(f"\n📊 Class weights:")
    for cls, weight in class_weight_dict.items():
        risk_name = {0: 'LOW', 1: 'INTERMEDIARY', 2: 'HIGH'}.get(cls, f'Class {cls}')
        print(f"   {risk_name}: {weight:.2f}")
    
    X_final = X
    y_final = y

print("\n✅ Block 8 complete!")
print(f"📊 Final features: {X_final.shape[1]}")
print(f"📊 Final target distribution:\n{pd.Series(y_final).value_counts().sort_index()}")


HANDLING CLASS IMBALANCE

🔍 Checking for string columns in features...
⚠️ Found string columns: ['Sex', 'Blood Pressure (mmHg)', 'Smoking Status', 'Diabetes Status', 'Physical Activity Level', 'Family History of CVD', 'Blood Pressure Category']
   These need to be encoded before SMOTE!
   ✅ Encoded: Sex
   ✅ Encoded: Blood Pressure (mmHg)
   ✅ Encoded: Smoking Status
   ✅ Encoded: Diabetes Status
   ✅ Encoded: Physical Activity Level
   ✅ Encoded: Family History of CVD
   ✅ Encoded: Blood Pressure Category

🔧 Recreating target encoding...

🔧 Checking for duplicate columns...
   ✅ No duplicate columns found

🔧 Preparing features and target...

📊 Features shape: (1309, 40)
📊 Target shape: (1309,)

📊 Target distribution:
CVD_Risk_Encoded
1    581
2    728
Name: count, dtype: int64

✅ SMOTE is available

📊 Minimum class count: 581

🔧 Applying SMOTE...

✅ After SMOTE:
   Shape: (1456, 40)
   Target distribution:
CVD_Risk_Encoded
1    728
2    728
Name: count, dtype: int64

✅ Block 8 comple

In [32]:
# ============================================
# BLOCK 9: SAVE PROCESSED DATA
# ============================================

print("\n" + "="*60)
print("SAVING PROCESSED DATA")
print("="*60)

# Create processed folder
processed_path = '../data/processed/'
os.makedirs(processed_path, exist_ok=True)

# Save original processed data
df.to_csv(f'{processed_path}processed_data_original.csv', index=False)
print(f"✅ Saved: processed_data_original.csv")
print(f"   Shape: {df.shape}")

# Save clean version
df_clean = pd.DataFrame(X, columns=feature_cols)
df_clean['CVD_Risk_Encoded'] = y
df_clean.to_csv(f'{processed_path}processed_data_clean.csv', index=False)
print(f"✅ Saved: processed_data_clean.csv")
print(f"   Shape: {df_clean.shape}")

# Save SMOTE version if applied
if SMOTE_AVAILABLE and len(y.unique()) >= 2 and y.value_counts().min() >= 2:
    df_resampled = pd.DataFrame(X_resampled, columns=feature_cols)
    df_resampled['CVD_Risk_Encoded'] = y_resampled
    df_resampled.to_csv(f'{processed_path}processed_data_smote.csv', index=False)
    print(f"✅ Saved: processed_data_smote.csv")
    print(f"   Shape: {df_resampled.shape}")

# Save the scaler
import joblib
joblib.dump(scaler, f'{processed_path}scaler.pkl')
print(f"✅ Saved: scaler.pkl")

print("\n" + "="*60)
print("FEATURE ENGINEERING COMPLETE! ✅")
print("="*60)
print("\n📁 Files saved in: data/processed/")
print("   - processed_data_original.csv")
print("   - processed_data_clean.csv")
if SMOTE_AVAILABLE and len(y.unique()) >= 2 and y.value_counts().min() >= 2:
    print("   - processed_data_smote.csv")
print("   - scaler.pkl")


SAVING PROCESSED DATA
✅ Saved: processed_data_original.csv
   Shape: (1309, 42)
✅ Saved: processed_data_clean.csv
   Shape: (1309, 41)
✅ Saved: processed_data_smote.csv
   Shape: (1456, 41)
✅ Saved: scaler.pkl

FEATURE ENGINEERING COMPLETE! ✅

📁 Files saved in: data/processed/
   - processed_data_original.csv
   - processed_data_clean.csv
   - processed_data_smote.csv
   - scaler.pkl
